# TriageIQ - 04 MLflow Experiment Tracking
Log several baseline experiments, compare them in the MLflow UI, and register the best model.

Prereq: `pip install mlflow`. Run from the `notebooks/` folder. Needs `../data/311_requests.csv`.

## 1. Setup

In [ ]:
import pandas as pd, numpy as np, mlflow, mlflow.sklearn, os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score, accuracy_score

mlflow.set_tracking_uri("sqlite:///../mlflow.db")
mlflow.set_experiment("triageiq-category-classifier")

df = pd.read_csv("../data/311_requests.csv", parse_dates=["created_date"])
df["description"] = df["description"].fillna("").str.lower().str.strip()
df["neighborhood"] = df["neighborhood"].fillna("Unknown")
df["hour"] = df.created_date.dt.hour.astype(str)
df["weekday"] = df.created_date.dt.weekday.astype(str)
cut = pd.Timestamp("2025-10-01")
train, test = df[df.created_date < cut], df[df.created_date >= cut]
y_train, y_test = train["category"], test["category"]
print(len(train), len(test))

## 2. Run experiments (each config is one MLflow run)

In [ ]:
def build(C, ngram, use_cat):
    cols = [("txt", TfidfVectorizer(ngram_range=ngram, min_df=2), "description")]
    if use_cat:
        cols.append(("cat", OneHotEncoder(handle_unknown="ignore"), ["neighborhood","channel","hour","weekday"]))
    return make_pipeline(ColumnTransformer(cols), LogisticRegression(C=C, max_iter=1000, class_weight="balanced"))

configs = [
    {"C": 0.5, "ngram": (1,1), "use_cat": False},
    {"C": 1.0, "ngram": (1,2), "use_cat": False},
    {"C": 1.0, "ngram": (1,2), "use_cat": True},
    {"C": 5.0, "ngram": (1,2), "use_cat": True},
]
results = []
for cfg in configs:
    with mlflow.start_run(run_name=f"lr_C{cfg['C']}_ng{cfg['ngram'][1]}_cat{cfg['use_cat']}") as run:
        model = build(**cfg).fit(train, y_train)
        pred = model.predict(test)
        f1 = f1_score(y_test, pred, average="macro")
        mlflow.log_params({"C": cfg["C"], "ngram_max": cfg["ngram"][1], "use_cat": cfg["use_cat"],
                           "split_cut": str(cut.date()), "train_rows": len(train)})
        mlflow.log_metrics({"macro_f1": f1, "accuracy": accuracy_score(y_test, pred)})
        mlflow.sklearn.log_model(model, "model")
        results.append({"run_id": run.info.run_id, **cfg, "macro_f1": round(f1, 4)})
pd.DataFrame(results)

## 3. Log the DistilBERT result (if notebook 03 was run)

In [ ]:
csv = "../models/model_comparison.csv"
if os.path.exists(csv):
    comp = pd.read_csv(csv)
    row = comp[comp.model == "DistilBERT"].iloc[0]
    with mlflow.start_run(run_name="distilbert_subsample"):
        mlflow.log_params({"model": "distilbert-base-uncased", "train_rows": 4000, "epochs": 2})
        mlflow.log_metrics({"macro_f1": float(row["macro_F1"]), "inference_sec_total": float(row["inference_sec_total"])})
    print("logged DistilBERT")
else:
    print("model_comparison.csv not found - skip")

## 4. Register the best sklearn model

In [ ]:
best = max(results, key=lambda r: r["macro_f1"])
print("best:", best)
mv = mlflow.register_model(f"runs:/{best['run_id']}/model", "triageiq-classifier")
print("registered version:", mv.version)

## 5. View the results in the MLflow UI
Open a terminal in the project root (not inside notebooks) and run:

`mlflow ui --backend-store-uri sqlite:///mlflow.db`

Then open http://127.0.0.1:5000, select the experiment, tick the runs, and click **Compare**. Take a screenshot for the README.

## 6. Findings (fill in)
- Best config and macro-F1:
- What changed the score most (n-grams, C, extra features)?
- Why tracking matters: reproducibility, comparison, rollback via the registry.